# Clase 8.1 — Protocolos Criptográficos: Zero-Knowledge Proofs (ZKP)

> **Curso:** Criptografía Aplicada  
> **Objetivo general:** Comprender los fundamentos de las pruebas de conocimiento cero, su seguridad formal, su implementación didáctica en Python y sus aplicaciones modernas.

---

## Tabla de contenidos

1. [Introducción](#1)
2. [Definición formal de ZKP](#2)
3. [Clases de ZKP](#3)
4. [Protocolo Sigma y Schnorr (interactivo)](#4)
5. [Práctica en Python: Schnorr completo](#5)
6. [Sonido y error de engaño (simulación)](#6)
7. [Transformación Fiat-Shamir (no interactiva)](#7)
8. [Práctica en Python: prueba no interactiva](#8)
9. [Aplicaciones reales](#9)
10. [Temas adicionales pertinentes](#10)
11. [Buenas prácticas de implementación](#11)
12. [Ejercicios propuestos](#12)
13. [Referencias](#13)


<a id='1'></a>
## 1) Introducción

Una **Zero-Knowledge Proof (ZKP)** permite que un *prover* convenza a un *verifier* de que una afirmación es verdadera **sin revelar información adicional** más allá de su validez.

En criptografía aplicada, las ZKP resuelven un dilema clásico: 

- Queremos autenticación o validación fuerte.
- Pero no queremos exponer secretos, datos privados o estructura interna del testigo (*witness*).

### Casos de uso típicos

- Identificación sin enviar contraseña ni secreto.
- Blockchains con privacidad (transacciones ocultas con validez pública).
- Pruebas de cumplimiento (*compliance proofs*) sin exponer datos sensibles.
- Sistemas de identidad y credenciales selectivas.


<a id='2'></a>
## 2) Definición formal de ZKP

Sea una relación \(R(x, w)\), donde \(x\) es la instancia pública y \(w\) el testigo secreto. Una prueba debe cumplir:

1. **Completitud (Completeness):**
   Si el *prover* honesto conoce \(w\), el *verifier* honesto acepta con probabilidad alta.

2. **Sonido (Soundness):**
   Si la afirmación es falsa, un *prover* malicioso no puede convencer al *verifier* salvo con probabilidad pequeña.

3. **Cero conocimiento (Zero-Knowledge):**
   Existe un simulador eficiente que puede generar transcripciones indistinguibles de una ejecución real, sin conocer \(w\).

### Intuición del simulador

Si todo lo que ve el verificador puede simularse sin secreto, entonces la interacción no filtra información útil sobre el testigo.


<a id='3'></a>
## 3) Clases de ZKP

- **Interactivas:** múltiples rondas reto-respuesta.
- **No interactivas (NIZK):** una sola prueba verificable sin interacción directa.
- **Argumentos sucintos (SNARK):** pruebas cortas y verificación rápida; suelen requerir supuestos algebraicos fuertes.
- **STARK:** transparencia (sin trusted setup), seguridad post-cuántica favorable, pruebas más grandes.
- **Bulletproofs:** pruebas cortas sin trusted setup para rangos y circuitos acotados.

### Parámetros que importan en práctica

- Tamaño de prueba.
- Tiempo de prueba y verificación.
- Necesidad de *trusted setup*.
- Hipótesis de seguridad (discrete log, pairings, hashes, etc.).


## Importaciones globales


In [ ]:
import secrets
import hashlib
import math
from dataclasses import dataclass

print('Entorno listo ✓')


<a id='4'></a>
## 4) Protocolo Sigma y Schnorr (interactivo)

Una familia central de ZKP son los **protocolos Sigma** (3 movimientos):

1. **Commitment:** el prover envía \(t\).
2. **Challenge:** el verifier envía reto \(c\).
3. **Response:** el prover responde \(s\).

### Schnorr Identification (conocimiento de log discreto)

Parámetros públicos: grupo \(G\), generador \(g\), orden \(q\).  
Secreto del prover: \(x\).  
Público asociado: \(y = g^x\).

Flujo:

- Prover elige \(r\) aleatorio y manda \(t = g^r\).
- Verifier manda reto \(c\).
- Prover responde \(s = r + c x \; (mod\; q)\).
- Verifier acepta si \(g^s = t \cdot y^c\).

> **Nota:** usaremos parámetros pequeños de juguete para aprendizaje.


In [ ]:
# Parámetros didácticos pequeños (NO producción)
p = 23
q = 11  # q divide p-1
g = 2   # elemento de orden q en este ejemplo

assert pow(g, q, p) == 1, 'g debe tener orden q (o divisor de q) en este ejemplo'

# Clave secreta y pública
x = secrets.randbelow(q - 1) + 1   # secreto del prover
y = pow(g, x, p)                    # clave pública

print(f'p={p}, q={q}, g={g}')
print('Secreto x (solo demo):', x)
print('Público y = g^x mod p:', y)


In [ ]:
@dataclass
class SchnorrTranscript:
    t: int
    c: int
    s: int


def schnorr_prove_interactive(x: int, p: int, q: int, g: int, c: int) -> SchnorrTranscript:
    """Genera una transcripción Schnorr para reto c en {0,1}."""
    if c not in (0, 1):
        raise ValueError('Este demo usa desafío binario c∈{0,1}')
    r = secrets.randbelow(q)
    t = pow(g, r, p)
    s = (r + c * x) % q
    return SchnorrTranscript(t=t, c=c, s=s)


def schnorr_verify(tr: SchnorrTranscript, y: int, p: int, g: int) -> bool:
    left = pow(g, tr.s, p)
    right = (tr.t * pow(y, tr.c, p)) % p
    return left == right

# Ejecución de una ronda honesta
c = secrets.randbelow(2)
tr = schnorr_prove_interactive(x, p, q, g, c)
print('Transcripción honesta:', tr)
print('Verificación:', schnorr_verify(tr, y, p, g))


<a id='5'></a>
## 5) Práctica en Python: Schnorr completo

En una autenticación real se repiten varias rondas para reducir la probabilidad de engaño.
Si el reto es de 1 bit, un atacante sin secreto tiene éxito aproximado de \(2^{-k}\) tras \(k\) rondas independientes.


In [ ]:
def run_honest_session(rounds: int = 16) -> bool:
    """Simula sesión interactiva completa de rounds rondas."""
    for _ in range(rounds):
        c = secrets.randbelow(2)
        tr = schnorr_prove_interactive(x, p, q, g, c)
        if not schnorr_verify(tr, y, p, g):
            return False
    return True

for k in [1, 4, 8, 16, 32]:
    ok = run_honest_session(k)
    print(f'Rondas={k:2d} -> aceptación honesta:', ok)


<a id='6'></a>
## 6) Sonido y error de engaño (simulación)

Un atacante que **no** conoce \(x\) puede intentar adivinar el reto \(c\) antes de verlo:

- Si adivina bien, fabrica una transcripción válida.
- Si no, falla.

Con desafío binario, su probabilidad por ronda es ~1/2.


In [ ]:
def inv_mod(a: int, p: int) -> int:
    return pow(a, -1, p)


def cheating_prover_one_round(y: int, p: int, g: int):
    """
    Atacante sin x: adivina c_hat y arma transcript válido solo para ese reto.
    Retorna (accepted, c_real, c_hat).
    """
    c_hat = secrets.randbelow(2)
    s = secrets.randbelow(q)

    if c_hat == 0:
        t = pow(g, s, p)
    else:
        t = (pow(g, s, p) * inv_mod(y, p)) % p

    # Verificador elige el reto real después de recibir t
    c_real = secrets.randbelow(2)
    tr = SchnorrTranscript(t=t, c=c_real, s=s)
    accepted = schnorr_verify(tr, y, p, g)
    return accepted, c_real, c_hat


def estimate_cheat_probability(rounds: int, trials: int = 5000) -> float:
    success = 0
    for _ in range(trials):
        ok = True
        for _ in range(rounds):
            accepted, _, _ = cheating_prover_one_round(y, p, g)
            if not accepted:
                ok = False
                break
        success += int(ok)
    return success / trials

for k in [1, 2, 4, 8, 12]:
    est = estimate_cheat_probability(k, trials=4000)
    theo = 2 ** (-k)
    print(f'k={k:2d} | éxito atacante estimado={est:.5f} | teórico~{theo:.5f}')


<a id='7'></a>
## 7) Transformación Fiat-Shamir (no interactiva)

Para evitar interacción, se reemplaza el reto aleatorio del verificador por un hash:

\[
 c = H(	ext{contexto} \parallel y \parallel t)
\]

Con esto obtenemos una prueba no interactiva (NIZK heurística en modelo de oráculo aleatorio).


In [ ]:
def hash_to_challenge(*parts: bytes, q: int) -> int:
    h = hashlib.sha256()
    for part in parts:
        h.update(part)
        h.update(b'|')
    return int.from_bytes(h.digest(), 'big') % q


def int_to_bytes(n: int) -> bytes:
    return n.to_bytes((n.bit_length() + 7) // 8 or 1, 'big')


def schnorr_prove_fiat_shamir(x: int, y: int, p: int, q: int, g: int, context: bytes = b'clase8.1'):
    r = secrets.randbelow(q)
    t = pow(g, r, p)
    c = hash_to_challenge(context, int_to_bytes(y), int_to_bytes(t), q=q)
    s = (r + c * x) % q
    return {'t': t, 's': s}


def schnorr_verify_fiat_shamir(proof: dict, y: int, p: int, q: int, g: int, context: bytes = b'clase8.1') -> bool:
    t = proof['t']
    s = proof['s']
    c = hash_to_challenge(context, int_to_bytes(y), int_to_bytes(t), q=q)
    left = pow(g, s, p)
    right = (t * pow(y, c, p)) % p
    return left == right

print('Funciones Fiat-Shamir listas ✓')


<a id='8'></a>
## 8) Práctica en Python: prueba no interactiva


In [ ]:
proof = schnorr_prove_fiat_shamir(x, y, p, q, g, context=b'login-demo')
print('Proof:', proof)
print('Verificación válida:', schnorr_verify_fiat_shamir(proof, y, p, q, g, context=b'login-demo'))

# Manipulación de la prueba
tampered_proof = dict(proof)
tampered_proof['s'] = (tampered_proof['s'] + 1) % q
print('Verificación tras modificar s:', schnorr_verify_fiat_shamir(tampered_proof, y, p, q, g, context=b'login-demo'))

# Mismo proof pero contexto distinto
print('Verificación en contexto distinto:', schnorr_verify_fiat_shamir(proof, y, p, q, g, context=b'otro-contexto'))


In [ ]:
# Mini escenario: autenticación por conocimiento de secreto sin revelarlo

def login_with_zkp(stored_public_y: int, proof: dict, context: bytes) -> bool:
    return schnorr_verify_fiat_shamir(proof, stored_public_y, p, q, g, context=context)

context = b'servicioA|nonce=12345'
proof_user = schnorr_prove_fiat_shamir(x, y, p, q, g, context=context)
accepted = login_with_zkp(y, proof_user, context)

print('Autenticación aceptada:', accepted)
print('El servidor solo conoce y y la prueba; no conoce x.')


<a id='9'></a>
## 9) Aplicaciones reales

1. **Privacidad en blockchain:**
   - zk-SNARKs en sistemas tipo Zcash.
   - zk-rollups para escalabilidad con pruebas de validez.

2. **Identidad y credenciales:**
   - Pruebas de atributos ("soy mayor de edad") sin revelar fecha exacta de nacimiento.

3. **Autenticación avanzada:**
   - Protocolos de identificación basados en log discreto o credenciales anónimas.

4. **Compliance y auditoría privada:**
   - Demostrar cumplimiento de reglas sin exponer toda la base de datos.


<a id='10'></a>
## 10) Temas adicionales pertinentes

### 10.1 Trusted Setup
Algunos SNARK requieren ceremonia inicial. Si se compromete, puede afectar seguridad. Diseños transparentes (STARK, Bulletproofs) evitan esta dependencia.

### 10.2 Recursión de pruebas
Permite verificar pruebas dentro de otras pruebas y construir agregación eficiente (útil en rollups).

### 10.3 Post-quantum
Las ZKP basadas en hashes/lattices son foco activo por su mejor perfil frente a amenazas cuánticas futuras.

### 10.4 UX y producto
En sistemas reales, la criptografía correcta no basta: manejo de nonces, sincronización, expiración de pruebas y protección anti-replay son críticos.


<a id='11'></a>
## 11) Buenas prácticas de implementación

1. Usar librerías auditadas y primitivas estándar.
2. Evitar parámetros "de juguete" fuera de laboratorio.
3. Incluir separación de dominio en hashes (`context`).
4. Usar nonces únicos y protección anti-replay.
5. Validar tamaños, rangos y pertenencia de grupo.
6. Diseñar protocolos con pruebas formales y revisión externa.
7. Medir costos (CPU, memoria, latencia, tamaño de prueba).
8. Documentar hipótesis de seguridad y modelo de amenaza.


<a id='12'></a>
## 12) Ejercicios propuestos

1. Modifica el tamaño del espacio de desafío y compara el error de sonido.
2. Implementa versión de Schnorr con reto de más de 1 bit y repite la simulación de atacante.
3. Añade un `nonce` de servidor obligatorio en el `context` y analiza replay.
4. Extiende la práctica a una prueba de conocimiento de representación (dos secretos).
5. Investiga diferencias prácticas entre Groth16, PLONK y STARK para un caso de uso real.
6. Diseña una arquitectura mínima de autenticación ZKP para una API.


<a id='13'></a>
## 13) Referencias

### Fundamentos y teoría

- Goldwasser, Micali, Rackoff (1985), *The Knowledge Complexity of Interactive Proof Systems*.
- Oded Goldreich, *Foundations of Cryptography, Vol. 1*.
- Katz & Lindell, *Introduction to Modern Cryptography*.

### Protocolos y estándares

- Claus-Peter Schnorr (1991), *Efficient Identification and Signatures for Smart Cards*.
- Fiat & Shamir (1986), *How to Prove Yourself: Practical Solutions to Identification and Signature Problems*.
- RFC 8235 — Schnorr NIZK Proof (contexto de uso en IETF).

### Recursos técnicos

- Zcash Protocol Specification: https://zips.z.cash/protocol/
- Electric Coin Company — ZK education resources: https://electriccoin.co/
- Ethereum Research (ZK Rollups): https://ethresear.ch/
- IACR ePrint Archive: https://eprint.iacr.org/

### Nota final

Este notebook es **didáctico**. Las implementaciones de producción deben usar bibliotecas especializadas, curvas/parámetros seguros y auditoría criptográfica formal.
